# Stage 11D data audit

This notebook presents the deterministic outputs created by `price_diffusion.stage11d`. It does not implement mapping, return, calendar, eligibility, or assembly logic. Run the Stage 11D module before refreshing this notebook.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DIAGNOSTICS = ROOT / 'outputs' / 'diagnostics'
panel = pd.read_parquet(ROOT / 'data' / 'processed' / 'daily_panel.parquet')
membership = pd.read_csv(ROOT / 'data' / 'processed' / 'universe_membership.csv')
mapping = pd.read_csv(DIAGNOSTICS / 'security_mapping_audit.csv')
coverage = pd.read_csv(DIAGNOSTICS / 'historical_coverage_audit.csv')
calendar = pd.read_csv(DIAGNOSTICS / 'trading_calendar_audit.csv')
exchange_calendar = pd.read_csv(DIAGNOSTICS / 'exchange_calendar_summary.csv')
quality = pd.read_csv(DIAGNOSTICS / 'return_quality_report.csv')

## Universe size and integration status

In [ ]:
summary = pd.Series({
    'approved_securities': len(mapping),
    'universe_version': panel['universe_version'].iloc[0],
    'securities_with_data': int(mapping['data_found'].sum()),
    'panel_rows': len(panel),
    'first_date': panel['date'].min(),
    'last_date': panel['date'].max(),
    'baseline_eligible_rows': int(panel['eligible'].sum()),
    'extension_eligible_rows': int(panel['extension_eligible'].sum()),
    'flagged_extreme_returns': int(panel['extreme_return_flag'].sum()),
}, name='value')
display(summary.to_frame())

## Historical coverage

In [ ]:
display(coverage.sort_values('number_of_observations').head(15))
coverage.set_index('ticker')['number_of_observations'].sort_values().plot(
    kind='barh', figsize=(9, 12), title='Daily observations by security'
)
plt.tight_layout()

## Missing dates and exchange calendars

The exchange summary counts weekdays without an observed local session. These include valid local holidays and closures and must not be filled automatically.

In [ ]:
display(exchange_calendar.sort_values('exchange'))
display(calendar.loc[calendar['missing_local_exchange_dates'].gt(0)])

## Problematic securities

In [ ]:
mapping_issues = mapping.loc[mapping['issues'].fillna('').ne('')]
coverage_issues = coverage.loc[coverage['issues'].fillna('').ne('')]
quality_issues = quality.loc[quality['issues'].fillna('').ne('')]
display(mapping_issues)
display(coverage_issues)
display(quality_issues)

## Historical baseline and extension eligibility

In [ ]:
eligibility_by_date = membership.groupby('date')[['eligible', 'extension_eligible']].sum()
display(eligibility_by_date.tail())
eligibility_by_date.plot(figsize=(11, 5), title='Eligible securities by date')
plt.ylabel('Security count')
plt.tight_layout()

## Example adjusted-price histories

In [ ]:
examples = ['NVDA', 'TSM', '000660.KS', 'ASML', '285A.T']
example_prices = panel.loc[panel['ticker'].isin(examples)].pivot(
    index='date', columns='ticker', values='adj_close'
)
example_prices.plot(figsize=(11, 6), logy=True, title='Example adjusted-price histories')
plt.ylabel('Adjusted close (log scale, local listing currency)')
plt.tight_layout()

## Interpretation boundary

The classification snapshot date is provenance; historical eligibility follows the separately configured eligibility window. Baseline and extension flags must not be substituted for one another. Extreme-return flags are diagnostic and do not delete observations.